# LeetCode #1065: Index Pairs of a String

https://leetcode.com/problems/index-pairs-of-a-string/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(\|text\| \cdot \|words\| \cdot L)$ | $O(1)$ |
| **Optimal: Trie ★** | $O(\sum L_i + \|text\| \cdot L_{max})$ | $O(\sum L_i)$ |

---

## Understanding the Methods

### Brute Force
For each word in `words`, scan `text` for all occurrences using substring matching. Simple but repeats work for shared prefixes.

### Optimal: Trie ★
Build a trie from all words. Then for each starting index in `text`, walk the trie character-by-character, recording an index pair whenever a trie node marks a complete word. Shared prefixes are traversed only once per starting position.

**Constraints:**
* $1 \le text.length \le 100$
* $1 \le words.length \le 20$, each word length $\le 50$

## Solutions

### C#

In [ ]:
using System.Collections.Generic;

public class Solution {
    // Trie node: children array + end-of-word flag
    class TrieNode {
        public TrieNode[] Children = new TrieNode[26];
        public bool IsWord;
    }

    public int[][] IndexPairs(string text, string[] words) {
        // Build trie from every word
        var root = new TrieNode();
        foreach (var w in words) {
            var node = root;
            foreach (var ch in w) {
                int idx = ch - 'a';
                node.Children[idx] ??= new TrieNode();
                node = node.Children[idx];
            }
            node.IsWord = true;
        }

        var result = new List<int[]>();
        // Try every starting index in text
        for (int i = 0; i < text.Length; i++) {
            var node = root;
            for (int j = i; j < text.Length; j++) {
                int idx = text[j] - 'a';
                if (node.Children[idx] == null) break; // no word with this prefix
                node = node.Children[idx];
                if (node.IsWord) result.Add(new[]{i, j}); // found a complete word
            }
        }
        // Result is already sorted (i increases, then j increases within same i)
        return result.ToArray();
    }
}

### Python

In [ ]:
class Solution:
    def indexPairs(self, text: str, words: list[str]) -> list[list[int]]:
        # Build a simple trie as nested dicts
        root = {}
        END = '#'
        for w in words:
            node = root
            for ch in w:
                node = node.setdefault(ch, {})
            node[END] = True  # mark complete word

        result = []
        n = len(text)
        for i in range(n):
            node = root
            for j in range(i, n):
                ch = text[j]
                if ch not in node:
                    break  # no word starts with text[i..j]
                node = node[ch]
                if END in node:
                    result.append([i, j])  # found matching word ending at j

        return result  # already sorted because i and j increase monotonically

### Go

In [ ]:
func indexPairs(text string, words []string) [][]int {
	// Build trie from all words
	type TrieNode struct {
		children [26]*TrieNode
		isWord   bool
	}
	root := &TrieNode{}
	for _, w := range words {
		node := root
		for _, ch := range w {
			idx := ch - 'a'
			if node.children[idx] == nil {
				node.children[idx] = &TrieNode{}
			}
			node = node.children[idx]
		}
		node.isWord = true
	}

	var result [][]int
	// Try every start position and walk the trie
	for i := 0; i < len(text); i++ {
		node := root
		for j := i; j < len(text); j++ {
			idx := text[j] - 'a'
			if node.children[idx] == nil {
				break // prefix not in trie
			}
			node = node.children[idx]
			if node.isWord {
				result = append(result, []int{i, j})
			}
		}
	}
	return result
}

### Rust

In [ ]:
impl Solution {
    pub fn index_pairs(text: String, words: Vec<String>) -> Vec<Vec<i32>> {
        // Build trie as a flat array of nodes
        let mut trie: Vec<[usize; 26]> = vec![[0usize; 26]];
        let mut is_word: Vec<bool> = vec![false];

        for w in &words {
            let mut node = 0usize;
            for ch in w.bytes() {
                let idx = (ch - b'a') as usize;
                if trie[node][idx] == 0 {
                    trie.push([0usize; 26]);
                    is_word.push(false);
                    trie[node][idx] = trie.len() - 1;
                }
                node = trie[node][idx];
            }
            is_word[node] = true; // mark complete word
        }

        let text = text.as_bytes();
        let mut result = Vec::new();
        for i in 0..text.len() {
            let mut node = 0usize;
            for j in i..text.len() {
                let idx = (text[j] - b'a') as usize;
                let next = trie[node][idx];
                if next == 0 { break; } // no match for this prefix
                node = next;
                if is_word[node] {
                    result.push(vec![i as i32, j as i32]);
                }
            }
        }
        result
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `text="thestoryofleetcodeandme", words=["story","fleet","leetcode"]`
Trie finds "story" at [3,7], "leetcode" at [13,20]. Result: `[[3,7],[13,20]]` in order.

### 2. Slightly Complex
**Input:** `text="abcd", words=["ab","bc","b","c"]`
At $i=1$: "b" matches at [1,1], "bc" matches at [1,2]. At $i=2$: "c" matches at [2,2]. Result: `[[1,1],[1,2],[2,2]]`.

### 3. Edge Case: Time Factor
**Input:** `text` has 100 characters all `'a'`; `words=["a","aa","aaa","aaaa","aaaaa"]`.
For each start $i$, the trie walk continues up to $\min(100-i, 5)$ steps, finding a word at every step. Total pairs $\approx 500$ — worst case for trie depth.

### 4. Edge Case: Space Factor
**Input:** 20 words of length 50, all sharing no prefix.
Trie has $20 \times 50 = 1000$ nodes — $O(\sum L_i)$ space. No node is shared, so the trie is widest possible.

### 5. Almost-Impossible but Plausible
**Input:** `text="a"*100`, `words=["a"*50]`.
Only one word of length 50. Trie walk from each $i$ terminates after 50 steps or at text end. Matches found at $[0,49],[1,50],\ldots,[50,99]$ — 51 pairs, proving the trie handles near-maximum overlap correctly.